# Check Real-Valued PCN
Small sanity check for `qvnn_pc.models.pcn` on a benchmark function.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
repo_root = next(path for path in candidates if (path / "src" / "qvnn_pc").exists())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

torch.set_printoptions(precision=4, sci_mode=True)
torch.manual_seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
from qvnn_pc.data import make_benchmark_regression_split
from qvnn_pc.models.pcn import PredictiveCodingNetwork

function_name = "ackley"
n = 30_000
sample_seed = 1
split_seed = 2

split = make_benchmark_regression_split(
    function_name,
    n=n,
    sample_seed=sample_seed,
    split_seed=split_seed,
    device=None,
)

# The benchmark split is quaternion-shaped for the quaternion models.
# For the plain PCN we use the real standardized inputs and scalar targets.
x_train = split.x_train[:, 0, :].to(device)
x_val = split.x_val[:, 0, :].to(device)
x_test = split.x_test[:, 0, :].to(device)

y_train = split.y_train_scalar.unsqueeze(-1).to(device)
y_val = split.y_val_scalar.unsqueeze(-1).to(device)
y_test = split.y_test_scalar.unsqueeze(-1).to(device)

x_train.shape, y_train.shape

In [ ]:
pcn = PredictiveCodingNetwork(
    dims=[4, 20, 20, 20, 20],
    output_dim=1,
    activation="tanh",
    prediction_inference_steps=32,
    prediction_eta_infer=0.05,
    device=device,
)

pcn

In [ ]:
probe_x = x_train[:8]
probe_y = y_train[:8]

probe_result = pcn.infer_latents(
    probe_x,
    y=probe_y,
    steps=8,
    eta_infer=0.05,
    return_trace=True,
)

probe_result["prediction"].shape, probe_result["trace"].keys()

In [ ]:
history = pcn.fit(
    x_train,
    y_train,
    x_val=x_val,
    y_val=y_val,
    epochs=100,
    batch_size=512,
    shuffle_generator=torch.Generator(device="cpu").manual_seed(1234),
    eta_infer=0.01,
    eta_learn=1e-4,
    T_infer=24,
    T_learn=1,
    eval_infer_steps=32,
    latent_init="feedforward",
)

history["runtime"], history["epochs_ran"]

In [ ]:
train_metrics = pcn.evaluate(x_train, y_train, infer_steps=32, eta_infer=0.05)
val_metrics = pcn.evaluate(x_val, y_val, infer_steps=32, eta_infer=0.05)
test_metrics = pcn.evaluate(x_test, y_test, infer_steps=32, eta_infer=0.05, return_trace=True)

pd.DataFrame(
    [train_metrics, val_metrics, {k: v for k, v in test_metrics.items() if k != "inference_trace"}],
    index=["train", "val", "test"],
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["mae"], label="train MAE")
axes[0].plot(history["validation_mae"], label="val MAE")
axes[0].set_title("Prediction Error")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MAE")
axes[0].legend()

axes[1].plot(history["free_energy"], label="train free energy")
axes[1].set_title("PC Objective")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Energy")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
trace = test_metrics["inference_trace"]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(trace["free_energy"])
axes[0].set_title("Inference Free-Energy Trace")
axes[0].set_xlabel("Inference step")
axes[0].set_ylabel("Energy")

axes[1].plot(trace["mean_latent_grad_norm"])
axes[1].set_title("Mean Latent Gradient Norm")
axes[1].set_xlabel("Inference step")
axes[1].set_ylabel("Gradient norm")

plt.tight_layout()
plt.show()

In [ ]:
pred = pcn.predict(x_test[:10], infer_steps=32, eta_infer=0.05)
pd.DataFrame(
    {
        "target": y_test[:10, 0].detach().cpu(),
        "prediction": pred[:10, 0].detach().cpu(),
    }
)

In [ ]:
# plot predicted values vs true values
fig = plt.figure(figsize=(10, 8))
with torch.no_grad():
    all_preds = pcn(x_test).cpu().squeeze(-1)
    all_true = y_test.cpu().squeeze(-1)

ax = fig.add_subplot(111)

ax.scatter(all_true, all_preds, s=8, alpha=0.5)

# Diagonale Linie
limits = [min(all_true.min(), all_preds.min()).item(), max(all_true.max(), all_preds.max()).item()]
ax.plot(limits, limits, color="black", linewidth=1, linestyle="--")

ax.set_title("Predicted vs True Values")
ax.set_xlabel("True")
ax.set_ylabel("Predicted")

plt.tight_layout()
plt.show()

In [ ]:
n_points_per_dim = 50
low, high = BOUNDS[function_name]
x1 = torch.linspace(low, high, n_points_per_dim)
x2 = torch.linspace(low, high, n_points_per_dim)
x3 = torch.linspace(low, high, n_points_per_dim)
x4 = torch.linspace(low, high, n_points_per_dim)
X1, X2, X3, X4 = torch.meshgrid(x1, x2, x3, x4, indexing="ij")
X_grid = torch.stack([X1, X2, X3, X4], dim=-1)
y_grid = FUNCTIONS[function_name](X_grid)

# Nur die Dimensionen 0 (x1) und 1 (x2) betrachten
d1, d2 = 0, 1
mid_idx = n_points_per_dim // 2

fig, ax = plt.subplots(figsize=(8, 6))

# Slice-Logik: x1 und x2 voll, x3 und x4 fixiert auf mid_idx
idx = [mid_idx] * 4
idx[d1] = slice(None)
idx[d2] = slice(None)

# Daten extrahieren
X = X_grid[tuple(idx + [d1])].cpu()
Y = X_grid[tuple(idx + [d2])].cpu()
Z = y_grid[tuple(idx)].cpu()

# Plot Landscape
cp = ax.contourf(X, Y, Z, levels=50, cmap='viridis')
fig.colorbar(cp, ax=ax, label='Funktionswert')

# Plot Scatter (Testpunkte)
x_test_fixed = split.raw_x_test.to(device)
pred = model.forward(x_test_fixed).detach().cpu().squeeze(-1)

ax.scatter(
    x_test_fixed[:, d1].cpu(), 
    x_test_fixed[:, d2].cpu(), 
    c=pred, 
    s=20, 
    edgecolors='white', 
    linewidth=0.5
)

ax.set_xlabel("x1")
ax.set_ylabel("x2")
ax.set_title(f"Schnitt: x1 vs x2 (x3, x4 fixiert auf Index {mid_idx})")

plt.show()